In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark

In [7]:
spark.createDataFrame(
    [
        (101, 'c1', 'IN', 100, '2024-01-01'),
        (102, 'c1', 'IN', 200, '2024-01-02'),
        (103, 'c2', 'US', 50, '2024-01-01'),
        (104, 'c3', 'US', 500, '2024-01-03'),
        (105, 'c2', 'US', 70, '2024-01-04'),
    ],
    ['order_id', 'customer_id', 'country', 'amount', 'order_date'],
).createOrReplaceTempView('orders')

# A1. Total amount and count by country.



In [9]:
result_a1 = spark.sql("""
    SELECT
        country,
        SUM(amount) AS total_amount,
        COUNT(order_id) AS order_count
    FROM orders
    GROUP BY country
    ORDER BY country
""")

result_a1.show()

+-------+------------+-----------+
|country|total_amount|order_count|
+-------+------------+-----------+
|     IN|         300|          2|
|     US|         620|          3|
+-------+------------+-----------+



# A2. Customers With Total >= 200 — CTE + HAVING

In [10]:
result_a2 = spark.sql("""
    WITH customer_totals AS (
        SELECT
            customer_id,
            SUM(amount) AS total_amount
        FROM orders
        GROUP BY customer_id
    )
    SELECT
        customer_id,
        total_amount
    FROM customer_totals
    WHERE total_amount >= 200
    ORDER BY customer_id
""")

result_a2.show()

+-----------+------------+
|customer_id|total_amount|
+-----------+------------+
|         c1|         300|
|         c3|         500|
+-----------+------------+



# A3. Orders Above Overall Average Amount

In [11]:
result_a3 = spark.sql("""
    SELECT
        order_id,
        customer_id,
        country,
        amount,
        order_date
    FROM orders
    WHERE amount > (
        SELECT AVG(amount)
        FROM orders
    )
    ORDER BY order_id
""")

result_a3.show()

+--------+-----------+-------+------+----------+
|order_id|customer_id|country|amount|order_date|
+--------+-----------+-------+------+----------+
|     102|         c1|     IN|   200|2024-01-02|
|     104|         c3|     US|   500|2024-01-03|
+--------+-----------+-------+------+----------+



# A4. WHERE vs HAVING example.
                ORDERS
                   |
                   v
               WHERE
           Filter rows
                   |
                   v
              GROUP BY
                   |
                   v
             AGGREGATION
          SUM / COUNT / AVG
                   |
                   v
               HAVING
           Filter groups
                   |
                   v
                RESULT

In [15]:
result_where = spark.sql("""
    SELECT
        country,
        SUM(amount) AS total_amount
    FROM orders
    WHERE amount > 100
    GROUP BY country
""")

result_where.show()


+-------+------------+
|country|total_amount|
+-------+------------+
|     IN|         200|
|     US|         500|
+-------+------------+



In [16]:
result_having = spark.sql("""
    SELECT
        country,
        SUM(amount) AS total_amount
    FROM orders
    GROUP BY country
    HAVING SUM(amount) > 300
""")

result_having.show()

+-------+------------+
|country|total_amount|
+-------+------------+
|     US|         620|
+-------+------------+

